In [85]:
import regex as re

In [156]:
with open('data.txt', 'r', encoding='utf-8', errors='ignore') as f:
    text = f.read()

In [157]:
len(text)

173898

In [155]:
class Tokenizer:

    def __init__(self):
        self.merges = {}
        self.vocab = {}

    def get_stats(self, tokens, counts = None):
        pair_count = counts if counts is not None else {}
        for t1, t2 in zip(tokens, tokens[1:]):
            pair_count[(t1, t2)] = pair_count.get((t1,t2), 0) + 1
        return pair_count
    
    def merge(self, tokens, max_pair, new_index):
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == max_pair[0] and tokens[i+1] == max_pair[1]:
                new_tokens.append(new_index)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens
    
    def apply_regrex(self, text):
        gpt2pat = re.compile(r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+""")
        tokens = re.findall(gpt2pat, text)
        tokens = [list(ch.encode('utf-8')) for ch in tokens]
        return tokens
    
    def train(self, text, new_vocab_size):
        tokens = self.apply_regrex(text)
        self.merges = {}
        for i in range (new_vocab_size - 256):
            stats = {}
            new_tokens = []
            for token in tokens:
                self.get_stats(token, stats)

            if not stats:
                break
            max_pair = max(stats, key=stats.get)
            new_index = i + 256
            self.merges[max_pair] = new_index

            for token in tokens:
                new_tokens.append(self.merge(token, max_pair, new_index))
            tokens = new_tokens
            print(f"Training done for epoch {i}")

        self.vocab = {idx: bytes([idx]) for idx in range(256)}
        for (po, p1), idx in self.merges.items():
            self.vocab[idx] = self.vocab[po] + self.vocab[p1]

    def encode_chunk(self, tokens):
        while len(tokens) > 1:
            stats = self.get_stats(tokens)

            pair = min(
                (pair for pair in stats if self.merges.get(pair)),
                key= lambda p: self.merges[p],
                default=None
            )

            if not pair:
                return tokens
            
            new_index = self.merges[pair]
            tokens = self.merge(tokens, pair, new_index)
        return tokens

    def encode(self, text):
        tokens = self.apply_regrex(text)
        ids = []

        for token in tokens:
            chunk_ids = self.encode_chunk(token)
            ids.extend(chunk_ids)

        return ids
    

    def decode(self, tokens):
        tokens = b''.join(self.vocab[token] for token in tokens)
        text = tokens.decode("utf-8", errors="replace")
        return text

In [ ]:
tokenizer1 = Tokenizer()
tokenizer1.train(text, new_vocab_size=30000)

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = enc.decode(ids) 
print(ids, text)

In [ ]:
ids = tokenizer1.encode("hello world!!!? (안녕하세요!) lol123 😉")
text = tokenizer1.decode(ids) 
print(ids, text)